# Day 7

# Helper Function

In [1]:
def process_raw_input(raw_history: str) -> list[list[str]]:
	result = []
	current_ls = None
	for line in raw_history.split("\n"):
		if line[0] == "$":
			if current_ls is not None:
				result.append(["ls", current_ls])
				current_ls = None
			command = line[2:4]
			arguments = line [5:]
			if command == "cd":
				result.append([command, arguments])
		else:
			if current_ls == None:
				current_ls = []
			current_ls.append(line)
	if current_ls is not None:
		result.append(["ls", current_ls])
		current_ls = None
	return result

In [2]:
class Folder:
	def __init__(self, name: str, parent: "Folder") -> None:
		self.name = name
		self.parent = parent
		self.folders = []
		self.files = []

	def add_new_folder(self, name: str, parent: "Folder") -> None:
		self.folders.append(Folder(name, parent))

	def add_file(self, name: str, size: int) -> None:
		self.files.append(File(name, self, size))

	def has_folder(self, check_folder: str) -> bool:
		for folder in self.folders:
			if folder.name == check_folder:
				return True
		return False

	def get_folder(self, check_folder: str) -> list["Folder"]:
		for folder in self.folders:
			if folder.name == check_folder:
				return folder
		return None

	def get_folders(self) -> list["Folder"]:
		return [folder.name for folder in self.folders]

	def get_size(self) -> int:
		size = 0
		for file in self.files:
			size += file.size
		for folder in self.folders:
			size += folder.get_size()
		return size

	def __str__(self) -> str:
		return f"- {self.name} (dir, size={self.get_size()})"

In [3]:
class File:
	def __init__(self, name: "str", folder: "Folder", size: int) -> None:
		self.name = name
		self.folder = folder
		self.size = size

	def __str__(self) -> str:
		return f"- {self.name} (file, size={self.size}"

In [4]:
def print_folder(folder: Folder, indent = 0) -> None:
	prefix = " " * (indent * 2)
	print(f"{prefix}{folder}")
	for subfolder in folder.folders:
		print_folder(subfolder, indent + 1)
	for file in folder.files:
		print(f"  {prefix}{file})")

In [5]:
def process_cd(argument: str, current_folder: Folder, root_folder: Folder):
	
	if argument == "/":
		current_folder = root_folder
	elif argument == "..":
		if current_folder is not root_folder:
			current_folder = current_folder.parent
	else:
		if current_folder.has_folder(argument):
			current_folder = current_folder.get_folder(argument)
		else:
			current_folder = Folder(argument, current_folder)

	return current_folder

In [6]:
def get_directory(history: list[str]) -> Folder:
	root_folder = Folder("/", None)
	current_folder = root_folder

	for line in history:
		match line[0]:
			case "ls":
				for item in line[1]:
					# print(item)
					if item[:4] == "dir ":
						if not current_folder.has_folder(item[:4]):
							current_folder.add_new_folder(item[4:], current_folder)
					else:
						size, name = item.split()
						current_folder.add_file(name, int(size))
			case "cd":
				current_folder = process_cd(line[1], current_folder, root_folder)

	return root_folder

In [7]:
def get_directory_summary(root_folder: Folder, directory_dict: dict | None = None, path: str = "") -> list[Folder]:
	if directory_dict is None:
		directory_dict = {}
		directory_dict["/"] = root_folder.get_size()
	for folder in root_folder.folders:
		directory_dict[path + folder.name] = folder.get_size()
		directory_dict = get_directory_summary(folder, directory_dict, folder.name + ".")

	return directory_dict

In [8]:
def get_delete_candidate(directory: list[Folder], total_disk: int, free_needed: int) -> str:
	directory_summary = get_directory_summary(directory)
	candidate_dirs = {k: v for (k,v) in directory_summary.items() if v >= directory.get_size() - (total_disk - free_needed)}

	results = [k for k in candidate_dirs if candidate_dirs[k] == min(candidate_dirs.values())]

	if len(results) > 0:
		return results[0]
	else:
		return None

# Unit Tests

In [9]:
import unittest

class TestNotebook(unittest.TestCase):
	test_input = """$ cd /
$ ls
dir a
14848514 b.txt
8504156 c.dat
dir d
$ cd a
$ ls
dir e
29116 f
2557 g
62596 h.lst
$ cd e
$ ls
584 i
$ cd ..
$ cd ..
$ cd d
$ ls
4060174 j
8033020 d.log
5626152 d.ext
7214296 k"""

	def setUp(self):
		self.history = process_raw_input(self.test_input)

	def test_has_folder(self):
		root_folder = Folder("/", None)
		current_folder = root_folder
		self.assertFalse(root_folder.has_folder("a"))
		
		current_folder.add_new_folder("a", current_folder)
		self.assertTrue(root_folder.has_folder("a"))
		new_folder = current_folder.get_folder("a")
		self.assertEqual(new_folder.parent, root_folder)

	def test_process_cd_command(self):
		root_folder = Folder("/", None)
		current_folder = root_folder

		current_folder = process_cd("a", current_folder, root_folder)
		self.assertEqual(current_folder.name, "a")

		current_folder = process_cd("b", current_folder, root_folder)
		self.assertEqual(current_folder.name, "b")

		current_folder = process_cd("c", current_folder, root_folder)
		self.assertEqual(current_folder.name, "c")

		current_folder = process_cd("d", current_folder, root_folder)
		self.assertEqual(current_folder.name, "d")

		current_folder = process_cd("..", current_folder, root_folder)
		self.assertEqual(current_folder.name, "c")

		current_folder = process_cd("/", current_folder, root_folder)
		self.assertEqual(current_folder.name, "/")

	def test_get_directory(self):
		directory = get_directory(self.history)
		dir_a = directory.get_folder("a")
		dir_e = dir_a.get_folder("e")
		dir_d = directory.get_folder("d")
		self.assertEqual(dir_e.get_size(), 584)
		self.assertEqual(dir_a.get_size(), 94853)
		self.assertEqual(dir_d.get_size(), 24933642)
		self.assertEqual(directory.get_size(), 48381165)

	def test_get_delete_candidate(self):
		get_delete_candidate
		self.assertEqual(get_delete_candidate(get_directory(self.history), 70000000, 30000000), "d")

unittest.main(argv=[''], verbosity=2, exit=False)

test_get_delete_candidate (__main__.TestNotebook) ... ok
test_get_directory (__main__.TestNotebook) ... ok
test_has_folder (__main__.TestNotebook) ... ok
test_process_cd_command (__main__.TestNotebook) ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.002s

OK


# Puzzle Solve

In [10]:
import os

with open("input" + os.sep + "7.txt", "r", encoding="utf-8") as input_values:
	raw_history = input_values.read()

history = process_raw_input(raw_history)
directory = get_directory(history)
dir_summary = get_directory_summary(directory)
over9000 = sum([v for (k,v) in dir_summary.items() if v < 100000])
delete_candidate = get_delete_candidate(directory, 70000000, 30000000)

print(f"over 9000: {over9000}")
print(f"delete_candidate: {dir_summary[delete_candidate]}")

over 9000: 1449447 (1449447)
delete_candidate: 8679207
